# Train the translation baseline

`tgt_input` starts with `<bos>`; `tgt_output` is the same sequence shifted one token forward and ends with `<eos>`. Training always feeds the reference token (100% Teacher Forcing). Cross-entropy ignores `<pad>`. PyTorch autograd performs backpropagation through the manually unrolled RNN. Clip the total gradient norm to 1.0. Select the checkpoint using dev loss, and keep test data untouched until selection is complete.

Default training is substantial: 1,966 train batches per epoch. The cell below shows a two-batch smoke run; remove the limits for the full run.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import torch
from src.data import load_corpus, make_loaders
from src.model import ModelConfig, Seq2Seq
from src.train import fit
corpus = load_corpus(PROJECT_ROOT / 'data')
train_loader, dev_loader, _ = make_loaders(corpus, batch_size=64)
src, lengths, tgt_input, tgt_output = next(iter(train_loader))
assert tgt_input.shape == tgt_output.shape
print('Batch shapes:', src.shape, lengths.shape, tgt_input.shape, tgt_output.shape)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Seq2Seq(ModelConfig(len(corpus.en_vocab), len(corpus.vi_vocab)))
# Educational smoke run; use src.train CLI without limits for the complete baseline.
fit(model, train_loader, dev_loader, PROJECT_ROOT / 'checkpoints' / 'smoke.pt', device, max_epochs=1, limit_train_batches=2, limit_dev_batches=1)